## DO 3 (LoRA): >10 labeled images - full LoRA training

## Install Packages

In [ ]:
# !uv pip install -q syft-flwr  # should have syft-client installed as dependency
!uv pip install -v "git+https://github.com/OpenMined/syft-flwr.git@main" 2>&1 | grep -E "(OpenMined/syft-flwr|OpenMined/syft-client).*[0-9a-f]{7}"

## Login

In [ ]:
import syft_client as sc
import syft_flwr

print(f"{sc.__version__ = }")
print(f"{syft_flwr.__version__ = }")

# do_email = input("Enter the Data Owner's email: ")
do_email = "dknguyen.buy@gmail.com"
do_client = sc.login_do(email=do_email)

## Check Connected Peers

In [ ]:
do_client.peers

## Create Syft Dataset

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download


# download the dataset from huggingface
DATASET_DIR = Path("./dataset/").expanduser().absolute()

if not DATASET_DIR.exists():
    snapshot_download(
        repo_id="khoaguin/chest-ct-segmentation",
        repo_type="dataset",
        allow_patterns="do3_lora/**",
        local_dir=DATASET_DIR,
    )

In [ ]:
# create a syft dataset (with mock and private path)
DATASET_PATH = DATASET_DIR / f"do3_lora"

do_client.create_dataset(
    name="chest-ct-segmentation",
    mock_path=DATASET_PATH / "mock",
    private_path=DATASET_PATH / "private",
    summary="Chest CT segmentations with images and masks",
    readme_path=DATASET_PATH / "README.md",
    sync=True,
)

do_client.datasets.get_all()

## DO Reviews, Approves and Runs Jobs

In [ ]:
do_client.jobs

In [ ]:
do_client.jobs[0].approve()

In [ ]:
do_client.process_approved_jobs()

## Clean Up

In [ ]:
do_client.delete_syftbox()